# 🛠️ Notebook 2 · Restaurant — Implementation

We now turn the design from Notebook 1 into real Python code. Every cell
**runs** and ends with a tiny `assert` so you can see it works.

Covered here:

1. Domain types: `MenuItem`, `Menu`, `Table`
2. Orders with a **state machine** (`OPEN → PLACED → SERVED → PAID`)
3. Billing with tax & tip
4. A `Staff` class hierarchy (Waiter / Chef / Manager)
5. A `Kitchen` that receives placed orders
6. `Reservation` for future bookings
7. End-to-end dinner scenario you can replay


## 🛠️ Setup

```bash
cd 07-object-oriented-design/restaurant
uv sync
```

Select the `.venv` kernel (top-right). If it's missing: `Cmd+Shift+P` →
**Reload Window**.


## 1️⃣ Menu and MenuItem

`MenuItem` is a `@dataclass(frozen=True)` — once created it can't be mutated.
That matches reality: you don't change the price of "Margherita" on an open
order. If the price changes tomorrow, you create a *new* `MenuItem`.


In [1]:
from dataclasses import dataclass, field
from enum import Enum
from itertools import count
from typing import Callable


@dataclass(frozen=True)
class MenuItem:
    name: str
    price: float
    category: str  # "pizza", "drinks", "dessert", ...
    vegetarian: bool = False


class Menu:
    """A small in-memory menu indexed by name."""

    def __init__(self, items: list[MenuItem]):
        self._items: dict[str, MenuItem] = {i.name: i for i in items}

    def find(self, name: str) -> MenuItem:
        if name not in self._items:
            raise KeyError(f"{name!r} is not on the menu")
        return self._items[name]

    def by_category(self, category: str) -> list[MenuItem]:
        return [i for i in self._items.values() if i.category == category]

    def vegetarian(self) -> list[MenuItem]:
        return [i for i in self._items.values() if i.vegetarian]

    def __iter__(self):
        return iter(self._items.values())


menu = Menu([
    MenuItem("Margherita", 10.0, "pizza", vegetarian=True),
    MenuItem("Pepperoni",  12.0, "pizza"),
    MenuItem("Caesar",      8.0, "salad"),
    MenuItem("Coke",        3.0, "drinks", vegetarian=True),
    MenuItem("Tiramisu",    6.0, "dessert", vegetarian=True),
])

print("Pizzas:", [p.name for p in menu.by_category("pizza")])
print("Veggie:", [p.name for p in menu.vegetarian()])

assert menu.find("Coke").price == 3.0
assert len(menu.vegetarian()) == 3


Pizzas: ['Margherita', 'Pepperoni']
Veggie: ['Margherita', 'Coke', 'Tiramisu']


## 2️⃣ Table and TableState

A table has a **state machine**: FREE ⇄ TAKEN, or FREE → RESERVED → TAKEN → FREE.
We enforce legal transitions with tiny methods instead of letting callers
overwrite `.state` directly.


In [2]:
class TableState(Enum):
    FREE = "free"
    TAKEN = "taken"
    RESERVED = "reserved"


@dataclass
class Table:
    number: int
    seats: int
    state: TableState = TableState.FREE

    def seat(self) -> None:
        if self.state not in (TableState.FREE, TableState.RESERVED):
            raise ValueError(f"table {self.number} is {self.state.value}")
        self.state = TableState.TAKEN

    def reserve(self) -> None:
        if self.state != TableState.FREE:
            raise ValueError(f"table {self.number} is {self.state.value}")
        self.state = TableState.RESERVED

    def free(self) -> None:
        self.state = TableState.FREE


t = Table(number=5, seats=4)
t.reserve()
assert t.state == TableState.RESERVED
t.seat()
assert t.state == TableState.TAKEN
t.free()
assert t.state == TableState.FREE
print("Table state machine works ✅")


Table state machine works ✅


## 3️⃣ Order lifecycle

`Order` owns a list of `OrderItem`s and an `OrderStatus`. The transitions are
`OPEN → PLACED → SERVED → PAID` — try to skip a step and you get a `ValueError`.

This is the single biggest win over the "god class": you cannot call `pay()`
on an order that was never served.


In [3]:
class OrderStatus(Enum):
    OPEN = "open"
    PLACED = "placed"
    SERVED = "served"
    PAID = "paid"


@dataclass
class OrderItem:
    item: MenuItem
    qty: int = 1
    notes: str = ""

    def line_total(self) -> float:
        return round(self.item.price * self.qty, 2)


_order_ids = count(1)  # simple auto-incrementing id generator


@dataclass
class Order:
    table: Table
    id: int = field(default_factory=lambda: next(_order_ids))
    items: list[OrderItem] = field(default_factory=list)
    status: OrderStatus = OrderStatus.OPEN
    waiter: "Waiter | None" = None

    # -- transitions -------------------------------------------------------
    def add(self, item: MenuItem, qty: int = 1, notes: str = "") -> None:
        self._require(OrderStatus.OPEN)
        if qty <= 0:
            raise ValueError("qty must be positive")
        self.items.append(OrderItem(item, qty, notes))

    def place(self) -> None:
        self._require(OrderStatus.OPEN)
        if not self.items:
            raise ValueError("cannot place an empty order")
        self.status = OrderStatus.PLACED

    def serve(self) -> None:
        self._require(OrderStatus.PLACED)
        self.status = OrderStatus.SERVED

    def pay(self, tax_rate: float = 0.08, tip_rate: float = 0.15) -> "Bill":
        self._require(OrderStatus.SERVED)
        self.status = OrderStatus.PAID
        self.table.free()
        return Bill.from_order(self, tax_rate, tip_rate)

    # -- helpers -----------------------------------------------------------
    def subtotal(self) -> float:
        return round(sum(i.line_total() for i in self.items), 2)

    def _require(self, expected: OrderStatus) -> None:
        if self.status != expected:
            raise ValueError(
                f"order {self.id} is {self.status.value}, needs {expected.value}"
            )


## 4️⃣ Bill — a small value object

`Bill` doesn't *do* much; it just remembers what was calculated. Keeping it
separate from `Order` means we can print, email, or archive bills without
dragging the whole order lifecycle along.


In [4]:
@dataclass(frozen=True)
class Bill:
    order_id: int
    subtotal: float
    tax: float
    tip: float
    total: float

    @classmethod
    def from_order(cls, order: Order, tax_rate: float, tip_rate: float) -> "Bill":
        sub = order.subtotal()
        tax = round(sub * tax_rate, 2)
        tip = round(sub * tip_rate, 2)
        return cls(
            order_id=order.id,
            subtotal=sub,
            tax=tax,
            tip=tip,
            total=round(sub + tax + tip, 2),
        )

    def pretty(self) -> str:
        return (
            f"--- Bill #{self.order_id} ---\n"
            f"Subtotal : ${self.subtotal:6.2f}\n"
            f"Tax      : ${self.tax:6.2f}\n"
            f"Tip      : ${self.tip:6.2f}\n"
            f"TOTAL    : ${self.total:6.2f}"
        )


## 5️⃣ Staff hierarchy

`Staff` is an **abstract base**. Each role (Waiter, Chef, Manager) inherits
and adds one behaviour. This is classic OOP polymorphism — and it's where
*Open/Closed* pays off: adding a `Sommelier` tomorrow costs nothing.


In [5]:
from abc import ABC, abstractmethod


@dataclass
class Staff(ABC):
    id: int
    name: str

    @abstractmethod
    def role(self) -> str: ...


@dataclass
class Waiter(Staff):
    def role(self) -> str:
        return "waiter"

    def take_order(self, table: Table) -> Order:
        if table.state != TableState.TAKEN:
            raise ValueError("can only take orders from seated tables")
        order = Order(table=table, waiter=self)
        return order


@dataclass
class Chef(Staff):
    def role(self) -> str:
        return "chef"

    def cook(self, order: Order) -> None:
        # In a real system the chef flips items from "queued" to "ready".
        # Here we just print what they're cooking.
        names = ", ".join(f"{oi.qty}x {oi.item.name}" for oi in order.items)
        print(f"👨‍🍳 Chef {self.name} is cooking order {order.id}: {names}")


@dataclass
class Manager(Staff):
    def role(self) -> str:
        return "manager"

    def comp_item(self, order: Order, item_name: str) -> None:
        """Manager privilege: remove an item from an open order (customer complaint)."""
        if order.status != OrderStatus.OPEN:
            raise ValueError("cannot comp a placed order")
        order.items = [oi for oi in order.items if oi.item.name != item_name]


alice  = Waiter(id=1, name="Alice")
bob    = Chef(id=2, name="Bob")
carla  = Manager(id=3, name="Carla")
for s in (alice, bob, carla):
    print(f"{s.name} is a {s.role()}")


Alice is a waiter
Bob is a chef
Carla is a manager


## 6️⃣ Kitchen — a tiny queue

When the waiter **places** an order, it goes to the `Kitchen`. The kitchen
keeps a FIFO queue and the chef pulls from it. Notice how `Kitchen` depends
only on `Order` — not on tables or waiters. Small, focused dependencies.


In [6]:
from collections import deque


class Kitchen:
    def __init__(self):
        self._queue: deque[Order] = deque()

    def receive(self, order: Order) -> None:
        if order.status != OrderStatus.PLACED:
            raise ValueError("kitchen only accepts placed orders")
        self._queue.append(order)

    def next_order(self) -> Order | None:
        return self._queue.popleft() if self._queue else None

    def pending(self) -> int:
        return len(self._queue)


## 7️⃣ Reservation — future bookings

A `Reservation` is a *promise* to hold a table for a guest at a time. When
the guest arrives, we flip the table RESERVED → TAKEN.


In [7]:
from datetime import datetime


@dataclass
class Reservation:
    table: Table
    guest_name: str
    when: datetime
    party_size: int

    def __post_init__(self):
        if self.party_size > self.table.seats:
            raise ValueError(
                f"table {self.table.number} only seats {self.table.seats}"
            )
        self.table.reserve()

    def check_in(self) -> None:
        self.table.seat()


t7 = Table(number=7, seats=2)
r = Reservation(t7, guest_name="Dana", when=datetime(2030, 1, 1, 19, 30), party_size=2)
assert t7.state == TableState.RESERVED
r.check_in()
assert t7.state == TableState.TAKEN
print("Reservation ✅")


Reservation ✅


## 8️⃣ End-to-end: one dinner, from seating to bill

Let's run the whole flow. Read the prints from top to bottom — it's exactly
what happens in a real restaurant.


In [8]:
# 1. Alice seats the guests at Table 5
table5 = Table(number=5, seats=4)
table5.seat()

# 2. Alice takes the order
order = alice.take_order(table5)
order.add(menu.find("Margherita"))
order.add(menu.find("Coke"), qty=2)
order.add(menu.find("Tiramisu"), notes="no dusting")

# 3. Carla (the manager) comps the tiramisu because the guest is a VIP
carla.comp_item(order, "Tiramisu")

# 4. Alice places the order; Kitchen queues it
kitchen = Kitchen()
order.place()
kitchen.receive(order)
assert kitchen.pending() == 1

# 5. Bob cooks the next order
current = kitchen.next_order()
bob.cook(current)

# 6. Food is served, guest pays
order.serve()
bill = order.pay()
print(bill.pretty())

# 7. Table is free again for the next guests
assert table5.state == TableState.FREE
print("\nTable 5 is free again →", table5.state.value)


👨‍🍳 Chef Bob is cooking order 1: 1x Margherita, 2x Coke
--- Bill #1 ---
Subtotal : $ 16.00
Tax      : $  1.28
Tip      : $  2.40
TOTAL    : $ 19.68

Table 5 is free again → free


## 9️⃣ Guardrails in action

Every invalid transition raises. That's how a tiny state machine buys you
safety. Try these on purpose:


In [9]:
def expect_error(fn):
    try:
        fn()
    except ValueError as e:
        print("✅ blocked:", e)
    else:
        print("❌ should have failed")


t = Table(number=9, seats=2)
t.seat()
bad_order = Order(table=t)

# can't place an empty order
expect_error(bad_order.place)

bad_order.add(menu.find("Coke"))
bad_order.place()

# can't pay before serving
expect_error(bad_order.pay)

# can't add items after placing
expect_error(lambda: bad_order.add(menu.find("Coke")))


✅ blocked: cannot place an empty order
✅ blocked: order 2 is placed, needs served
✅ blocked: order 2 is placed, needs open


## 🔟 Try it yourself

- Add a `happy_hour` flag on `MenuItem` and give drinks a 20% discount
  between 17:00–19:00. *(Hint: the cleanest way is a **Strategy** — see
  Notebook 3.)*
- Add `split_bill(n)` on `Order` that divides the total evenly by `n`.
- Add a `Receipt` printer that writes a text file for each paid bill.
- Model a *takeaway* order with no `Table` — what changes, what stays the
  same?

➡️ **Continue to Notebook 3** to meet the three patterns every OOD interview
asks about: Strategy, Factory, Observer.
